In [1]:
import os
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["PYVISTA_OFF_SCREEN"] = "True"
os.environ["PYVISTA_USE_WGL"] = "True"
from os import environ
environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
environ['CUDA_VISIBLE_DEVICES'] = '0' #Select the GPU Card
print('Process id: ',os.getpid())
# !pip install trimesh


import os, glob, csv, h5py, pickle, struct, cv2, imageio, timeit, sys, shutil
import numpy as np
#print(np.__version__)
import pandas as pd
import math as math
import scipy.io as sio
import scipy.stats as stats
import scipy.linalg as linalg
import scipy.integrate as integrate
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
import time as time
import trimesh
import open3d as o3d
import pyvista as pv
import pydicom
from pydicom.filereader import dcmread
from pydicom.pixel_data_handlers.util import apply_modality_lut


from os import environ, listdir
from os.path import dirname, abspath
from sys import argv
from numpy import zeros, newaxis
from random import randint, choice, shuffle, sample
from itertools import product
from scipy import signal
from scipy.io import loadmat
from scipy import ndimage, misc
from scipy.ndimage.interpolation import rotate, zoom
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.utils import class_weight
from cv2 import imread, imwrite
from PIL import Image, ImageOps
from skimage.exposure import rescale_intensity, adjust_gamma
from scipy.ndimage import map_coordinates, gaussian_filter
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from ipywidgets import interact, interactive, fixed, interact_manual
# from construct import PaddedString, Int16un, Struct, Int32un, Int8un, Array
from pathlib import Path


import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices('GPU'))
import tensorflow.keras
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model, Model
from tensorflow.keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D,\
 MaxPooling2D, ZeroPadding2D, UpSampling2D, concatenate, Average, add, Reshape, Permute, Conv2D,\
 Conv2DTranspose, SeparableConv2D, ELU, BatchNormalization, AveragePooling2D, Cropping2D, \
 Flatten, Dense, Dropout, Activation, SpatialDropout2D
from tensorflow.keras.optimizers import RMSprop, SGD, Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, LearningRateScheduler, Callback
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model, to_categorical, Sequence

from scipy.ndimage import zoom

from skimage import feature
from ipywidgets import interact, interactive, fixed 
from mpl_toolkits.mplot3d import Axes3D

import trimesh
from trimesh.voxel import VoxelGrid
import pyvista as pv
from contextlib import redirect_stdout

Process id:  327349
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/tmp/ipykernel_327349/3050048457.py:41: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  from scipy import ndimage, misc
/tmp/ipykernel_327349/3050048457.py:42: DeprecationWarning: Please import `rotate` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.interpolation import rotate, zoom
/tmp/ipykernel_327349/3050048457.py:42: DeprecationWarning: Please import `zoom` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.interpolation import rotate, zoom
2026-03-18 11:57:24.548419: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 11:57:24.5616

TensorFlow version: 2.18.0
GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
def DicomReader(dicom_folder_path):
    # --- Collect DICOM files ---
    dicom_files = [
        os.path.join(dicom_folder_path, f)
        for f in os.listdir(dicom_folder_path)
        if f.endswith(".dcm")
    ]
    if len(dicom_files) == 0:
        raise ValueError("No DICOM files found in folder.")

    # --- Pick the largest (likely multi-frame) DICOM ---
    dicom_files.sort(key=os.path.getsize, reverse=True)
    selected_file = dicom_files[0]
    if len(dicom_files) > 1:
        print(f"Multiple DICOMs found. Using the largest file: {os.path.basename(selected_file)}")

    dcm = dcmread(selected_file)

    # --- Get OCT volume ---
    oct_volume = apply_modality_lut(dcm.pixel_array, dcm)  # Shape: (frames, H, W)
    if oct_volume.ndim == 3:
        oct_volume = np.transpose(oct_volume, (1, 2, 0))  # (H, W, slices)
    elif oct_volume.ndim == 2:
        oct_volume = np.expand_dims(oct_volume, axis=-1)  # Single slice

    # --- Extract voxel spacing ---
    dx = float(getattr(dcm, "PixelSpacing", [1, 1])[0])
    dy = float(getattr(dcm, "PixelSpacing", [1, 1])[1])
    dz = float(getattr(dcm, "SpacingBetweenSlices", getattr(dcm, "SliceThickness", 1.0)))

    # --- Prepare output dict ---
    volData = {
        'OCTFrames': oct_volume,
        'voxelDimensions': [dx, dy, dz],
        'NumBScans': oct_volume.shape[2],
        'SizeY': oct_volume.shape[0],
        'SizeX': oct_volume.shape[1],
        'SizeXSlo': 0,
        'SizeYSlo': 0,
        'ScaleXSlo': 0,
        'ScaleYSlo': 0,
        'SLO': None,
        'BScanLines': np.zeros((oct_volume.shape[2], 2, 2)),
        'SegArray': np.zeros((oct_volume.shape[1], 17, oct_volume.shape[2])),
        'ScanPosition': getattr(dcm, 'PatientPosition', 'Unknown')
    }

    return volData
    
def process_label(label_file):
    """
    This fucntion takes label files with layers marked from 1 to 8, not 0 to 7
    """
    colors = np.zeros((9,3),dtype=np.uint8)
    
    # #### colors for each label edit accordingly
    # colors[0] = [0, 0, 0]
    # colors[1] = [255,0,0]
    # colors[2] = [0,255,0]
    # colors[3] = [0,0,255]
    # colors[4] = [255,255,0]
    # colors[5] = [255,0,255]
    # colors[6] = [0, 255, 255]
    # colors[7] = [255, 153, 51]
    # #colors[8] = [255, 255, 255]
    #### Define colors for each label
    colors[0] = [255, 255, 255]  # Background (white)
    colors[1] = [255, 0, 0]      # Class 1 (Red)
    colors[2] = [0, 255, 0]      # Class 2 (Green)
    colors[3] = [0, 0, 255]      # Class 3 (Blue)
    colors[4] = [0, 255, 255]     # Class 4 (cyan)
    colors[5] = [255, 165, 0]    # Class 5 (Orange)
    colors[6] = [150, 75, 0]    # Class 6 (Brown)
    colors[7] = [238, 130, 238]  # Class 7 (Violet)
    
    H = np.shape(label_file)[0]
    W = np.shape(label_file)[1]
    
   # img = np.zeros((H,W,3),dtype=np.uint8)
    img = np.full((H, W, 3), 255, dtype=np.uint8)  # White background for solid map
    # for i,j in product(range(H),range(W)):
    #     img[i, j] = colors[label_file[i,j]]
    for i in range(1, len(colors)):  # Assuming labels from 1 to 8
        img[label_file == i] = colors[i]  # Fill segmented areas with solid color
    
    return np.asarray(img)


def GetBscansFromVolFile(volData):
  # bscan_all = interpolate_data(vol_list['OCTFrames'], (496, 384, 97), (192, 536, 361))

    bscan_all = volData['OCTFrames']

    bscan_resize = []
    for x in range(np.shape(bscan_all)[2]):
        bscan = bscan_all[:,:,x]
        bscan[bscan > 255] = 0
        bscan1 = np.asarray(bscan, dtype=np.uint8)
        bscan_resize.append(bscan1)
    
    bscan_resize = np.asarray(bscan_resize,dtype = np.uint8)
    return bscan_resize


def NormalizeImage(img):
  if (np.max(img)-np.min(img)) != 0:
    norm_img = (img-np.min(img))/(np.max(img)-np.min(img))
  else:
    norm_img = np.copy(img)

  return  norm_img
   
def GetSegmentedBscans(bscan_all_):
  bscan_resize = []
  for x in range(np.shape(bscan_all_)[0]):
    bscan1 = np.asarray(Image.fromarray(np.asarray(bscan_all_[x, :,:],dtype = np.uint8)).resize((400,200)), dtype = np.uint8)
    bscan1 = NormalizeImage(bscan1)
    bscan_resize.append(bscan1)

  bscan_resize = np.asarray(bscan_resize)
  bscan_resize_exp = np.expand_dims(bscan_resize,axis=3)
  # # print(np.shape(bscan_resize_exp))
  # class DynamicCropLayer(Layer):
  #   def __init__(self, **kwargs):
  #       super(DynamicCropLayer, self).__init__(**kwargs)

  #   def call(self, inputs):  
  #       target_shape = tf.shape(inputs[1])  
  #       input_shape = tf.shape(inputs[0])  
  #       crop_height = (input_shape[1] - target_shape[1]) // 2
  #       crop_width = (input_shape[2] - target_shape[2]) // 2
        
  #       cropped = tf.slice(
  #           inputs[0],
  #           [0, crop_height, crop_width, 0],
  #           [-1, target_shape[1], target_shape[2], -1]
  #       )
  #       return cropped  

  # load model
  modelpath = './../../../../../mnt/DATA/sda1/Suraj/New directory/Model_checkpoints/'
  model = load_model(modelpath + 'Attention_UNet_cce_pf_weight_1.h5', compile = False)
  # model = load_model(modelpath + 'Dunet_cce.h5', 
  #                  custom_objects={'DynamicCropLayer': DynamicCropLayer}, 
  #                  compile=False)
  model.load_weights(modelpath+'best_weights_Attention_UNet_cce_pf_weight_1.weights.h5')
  # model = load_model(modelpath + 'OCT_segmentation_1_unetplus_cce.h5')
  # model.load_weights(modelpath+'best_weights_OCT_segmentation_1_unetplus_cce.weights.h5')
  # print('Model loaded...')    

  # get prediction
  bscan_preds = model.predict(bscan_resize_exp)
  # print(np.shape(bscan_preds))

  # convert segmnetation back into original resolution
  bscan_preds1=[]
  for x in range(len(bscan_preds)):
    temp1 = np.array( np.argmax(bscan_preds[x],axis=2) , dtype=np.uint8)
    bscan_preds1.append(temp1)
  bscan_preds1 = np.array(bscan_preds1)
  # print(np.shape(bscan_preds1))

  return bscan_resize, bscan_preds1 


def interpolate_data(data, data_shape, target_shape):
    # Compute zoom factors for each dimension
    zoom_factors = (target_shape[0] / data_shape[0],
                  target_shape[1] / data_shape[1],
                  target_shape[2] / data_shape[2])
    interpolated_data = zoom(data, zoom_factors)
    
    print("Interpolated shape:", interpolated_data.shape)
    
    return interpolated_data


def ExtractONHLayer(bscan_segmented_, layermarker):
    layer = []
    for i in range(len(bscan_segmented_)):
        bscan = np.copy(bscan_segmented_[i])
        bscan[bscan!=layermarker] = 0
        bscan[bscan==layermarker] = 1
    
        bscan = np.asarray(bscan,dtype = np.uint8)
        # bscan = np.asarray(Image.fromarray(bscan).resize((534,192)), dtype = np.uint8)
    
        layer.append(bscan)
    
    layer = np.array(layer)
    return layer



def ZoomNumpyVolume(numpyarr):
    # Compute the zoom factors
    zoom_factors = (400 /68, 1, 1)  # Scale only along the first axis
    
    # Apply zoom
    resized_arr = zoom(numpyarr, zoom_factors, order=3)  # Cubic interpolation (order=3)

    return resized_arr




def RnflEnfaceView(rnfl_layer_,rpe_layer_, dx,dz, dy):
  rnfl_thickness=[]
  for i in range(len(rnfl_layer_)):
    img = rnfl_layer_[i]
    thick = np.sum(img,axis =0)*dz*100
    rnfl_thickness.append(thick)

  rnfl_thickness = np.array(rnfl_thickness)

  rpe_thickness=[]
  for i in range(len(rpe_layer_)):
    img = rpe_layer_[i]
    thick = np.sum(img,axis =0) #um
    rpe_thickness.append(thick)

  rpe_thickness = np.array(rpe_thickness)
  rpe_thickness[rpe_thickness !=0] =1

  rnfl_thickness_new = rnfl_thickness #*rpe_thickness

  # rnfl_thickness_new = np.asarray(Image.fromarray(np.asarray(rnfl_thickness_new,dtype = np.uint8)).resize((400,200)), dtype = np.uint8)

  return rnfl_thickness_new


def GclEnfaceView(gcl_layer_, rpe_layer_, dx, dz, dy):
    gcl_thickness = []

    # Compute GCL thickness for each slice
    for img in gcl_layer_:
        thick = np.sum(img, axis=0) * dz*100  # Sum over depth, convert mm -> µm
        gcl_thickness.append(thick)

    gcl_thickness = np.array(gcl_thickness)

    # Create RPE mask
    rpe_thickness = []
    for img in rpe_layer_:
        thick = np.sum(img, axis=0)
        rpe_thickness.append(thick)

    rpe_thickness = np.array(rpe_thickness)
    rpe_thickness[rpe_thickness != 0] = 1  # Binary mask

    # Optionally apply the RPE mask to GCL thickness
    gcl_thickness_new = gcl_thickness

    return gcl_thickness_new

def OrlEnfaceView(orl_layer_, rpe_layer_, dx, dz, dy):
    orl_thickness = []

    # Compute ORL thickness for each slice
    for img in orl_layer_:
        thick = np.sum(img, axis=0) * dz *100  # Sum over depth, convert mm -> µm
        orl_thickness.append(thick)

    orl_thickness = np.array(orl_thickness)

    # Create RPE mask
    rpe_thickness = []
    for img in rpe_layer_:
        thick = np.sum(img, axis=0)
        rpe_thickness.append(thick)

    rpe_thickness = np.array(rpe_thickness)
    rpe_thickness[rpe_thickness != 0] = 1  # Binary mask

    # Optionally apply the RPE mask
    orl_thickness_new = orl_thickness  # * rpe_thickness

    return orl_thickness_new


def RpeEnfaceView(rpe_layer_, dx, dz, dy):
    rpe_thickness = []

    # Compute RPE thickness for each slice
    for img in rpe_layer_:
        thick = np.sum(img, axis=0) *dz*100  # Sum over depth, convert mm -> µm
        rpe_thickness.append(thick)

    rpe_thickness = np.array(rpe_thickness)

    # No need for another RPE mask here — RPE layer itself is what we want
    rpe_thickness_new = rpe_thickness

    # Optional resizing step (commented out)
    # rpe_thickness_new = np.asarray(Image.fromarray(np.asarray(rpe_thickness_new, dtype=np.uint8)).resize((400, 200)), dtype=np.uint8)

    return rpe_thickness_new


In [75]:
dicom_data = dcmread('/mnt/DATA/sda1/Suraj/AIIMSNagpur/Case/Case Group/S47 719730/onh/DESKTOP-B2D0PPH_AIIMS Nagpur_46044.689965277779_6dbb0000_90/46044.689965277779_6dbb0000_91.dcm')

In [76]:
# dicom_data

In [77]:
print(dicom_data['PixelData'])

img = np.array(dicom_data['PixelData'])
print((img))

(7FE0,0010) Pixel Data                          OB: Array of 13903872 elements
(7FE0,0010) Pixel Data                          OB: Array of 13903872 elements


In [78]:
if 'SharedFunctionalGroupsSequence' in dicom_data:
    shared_func_groups_seq = dicom_data.SharedFunctionalGroupsSequence
    if len(shared_func_groups_seq) > 0:
        # Check if Pixel Measures Sequence is inside the shared functional groups
        if 'PixelMeasuresSequence' in shared_func_groups_seq[0]:
            pixel_measures_sequence = shared_func_groups_seq[0].PixelMeasuresSequence
            if len(pixel_measures_sequence) > 0:
                pixel_spacing = pixel_measures_sequence[0].PixelSpacing
                print(f"Pixel Spacing: {pixel_spacing}")
            else:
                print("Pixel Measures Sequence is empty.")
        else:
            print("Pixel Measures Sequence not found in Shared Functional Groups Sequence.")
    else:
        print("Shared Functional Groups Sequence is empty.")
else:
    print("Shared Functional Groups Sequence not found in the DICOM file.")

Pixel Spacing: [0.003872, 0.011509]


In [79]:
# Extract the pixel data (assuming it's stored as a single continuous array)
pixel_data = dicom_data.pixel_array
# Function to display a specific frame
def display_frame(frame_idx):
    plt.figure(figsize=(12, 6))
    plt.imshow(pixel_data[frame_idx], cmap='gray')
    plt.title(f"Frame {frame_idx + 1}")
    plt.axis('on')
    plt.show()

# Create an interactive slider to navigate through frames
interact(display_frame, frame_idx=widgets.IntSlider(min=0, max=pixel_data.shape[0]-1, step=1, value=0));

interactive(children=(IntSlider(value=0, description='frame_idx', max=72), Output()), _dom_classes=('widget-in…

In [80]:
# pixel_data = dicom_data.pixel_array  # for en-face view

# # Remove singleton dimension if present
# if pixel_data.ndim == 3 and pixel_data.shape[-1] == 1:
#     image = pixel_data[:, :, 0]
# elif pixel_data.ndim == 2:
#     image = pixel_data
# else:
#     raise ValueError(f"Unexpected pixel_array shape: {pixel_data.shape}")

# print(image.shape)  # (768, 768)

# plt.figure(figsize=(6, 6))
# plt.imshow(image, cmap='gray')
# plt.title("OCT IR Image (768 × 768)")
# plt.axis('off')
# plt.show()

In [91]:
dicom_folder_path = "./../../../../../mnt/DATA/sda1/Suraj/AIIMSNagpur/Case/Case Group/S37  248562/onh/DESKTOP-B2D0PPH_AIIMS Nagpur_45968.679212962961_6e360000_46/"
dicom_files = [os.path.join(dicom_folder_path, f) for f in os.listdir(dicom_folder_path) if f.endswith(".dcm")]

print(f"Found {len(dicom_files)} DICOM files.")

for f in dicom_files[:5]:  # check first 5
    d = dcmread(f)
    print("\nFile:", os.path.basename(f))
    print("Rows x Cols:", d.Rows, "x", d.Columns)
    print("SeriesDescription:", getattr(d, "SeriesDescription", "N/A"))
    print("ImageType:", getattr(d, "ImageType", "N/A"))
    print("InstanceNumber:", getattr(d, "InstanceNumber", "N/A"))
    print("ImagePositionPatient:", getattr(d, "ImagePositionPatient", "N/A"))
    print("NumberOfFrames:", getattr(d, "NumberOfFrames", "N/A"))

dicom_path = "./../../../../..//mnt/DATA/sda1/Suraj/AIIMSNagpur/Case/Case Group/S37  248562/onh/DESKTOP-B2D0PPH_AIIMS Nagpur_45968.679212962961_6e360000_46/45968.679212962961_6e360000_47.dcm"
volData = DicomReader(os.path.dirname(dicom_path))

print("OCT Volume shape:", volData['OCTFrames'].shape)
print("Voxel dimensions:", volData['voxelDimensions'])

ds = pydicom.dcmread(dicom_path)
# Most OCT DICOMs have this:
fg = ds.SharedFunctionalGroupsSequence[0]
pixel_measures = fg.PixelMeasuresSequence[0]

# Try to get lateral (x), axial (y), and depth (z) spacing
px, py = [float(x) for x in pixel_measures.PixelSpacing]  # mm per pixel
tz = float(pixel_measures.SliceThickness)  # mm per slice

print(f"Lateral spacing (X): {px} mm")
print(f"Axial spacing (Y): {py} mm")
print(f"Slice spacing (Z): {tz} mm")

spacing = (tz, py, px)  # (z, y, x)
print("Final spacing tuple (z, y, x):", spacing)

Found 1 DICOM files.

File: 45968.679212962961_6e360000_47.dcm
Rows x Cols: 496 x 384
SeriesDescription: Volume IR
ImageType: ['ORIGINAL', 'PRIMARY']
InstanceNumber: 1
ImagePositionPatient: N/A
NumberOfFrames: 73
OCT Volume shape: (496, 384, 73)
Voxel dimensions: [1.0, 1.0, 1.0]
Lateral spacing (X): 0.003872 mm
Axial spacing (Y): 0.011322 mm
Slice spacing (Z): 0.060305 mm
Final spacing tuple (z, y, x): (0.060305, 0.011322, 0.003872)


In [92]:
bscans = GetBscansFromVolFile(volData)
print(np.shape(bscans))

(73, 496, 384)


In [93]:
@interact(x=(0,len(bscans)-1,1))
def h(x=0):
    f = plt.figure(figsize=(6,6))
    plt.imshow(bscans[x, :,:],cmap='gray')
    plt.show()

interactive(children=(IntSlider(value=0, description='x', max=72), Output()), _dom_classes=('widget-interact',…

In [94]:
bscans_resize, bscans_segmented = GetSegmentedBscans(bscans)
print(np.shape(bscans_resize), np.shape(bscans_segmented))

3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step   
(73, 200, 400) (73, 200, 400)


In [95]:
@interact(x=(0,len(bscans_segmented)-1,1))
def h(x=0):
    colorimage = process_label(bscans_segmented[x, :,:])
    
    f = plt.figure(figsize=(12,6))
    f.add_subplot(1,3,1)
    plt.imshow(bscans_resize[x, :,:],cmap='gray')
    plt.axis('off')

    f.add_subplot(1,3,2)
    plt.imshow(colorimage)
    plt.axis('off')

    f.add_subplot(1,3,3)
    plt.imshow(bscans_resize[x, :,:],cmap='gray')
    plt.imshow(colorimage, alpha = 0.25)
    plt.axis('off')

    plt.show()

interactive(children=(IntSlider(value=0, description='x', max=72), Output()), _dom_classes=('widget-interact',…

In [96]:
# dx, dy, dz = volData['voxelDimensions']  # from DICOM
dz, dy, dx = np.array(spacing)
print("dz, dy, dx:", dz,dy, dx)

dz, dy, dx: 0.060305 0.011322 0.003872


In [97]:
bscan_zoomed = interpolate_data(bscans_segmented, (73, 200, 400), (73, 200, 400))

Interpolated shape: (73, 200, 400)


In [98]:
@interact(x=(0,len(bscan_zoomed)-1,1))
def h(x=0):
    colorimage = process_label(bscan_zoomed[x, :,:])
    
    f = plt.figure(figsize=(6,6))
    plt.imshow(colorimage)
    plt.axis('off')

interactive(children=(IntSlider(value=0, description='x', max=72), Output()), _dom_classes=('widget-interact',…

In [99]:
rnfl = ExtractONHLayer(bscan_zoomed, 1)
print(np.unique(rnfl))
print(np.shape(rnfl))

[0 1]
(73, 200, 400)


In [100]:
# Function to remove small contours in a binary mask
def remove_small_contours(binary_mask, min_contour_area):
    cleaned_mask = np.zeros_like(binary_mask)
    for i in range(binary_mask.shape[0]):  # Iterate over slices
        slice_mask = (binary_mask[i] * 255).astype(np.uint8)  # Convert to uint8 for OpenCV

        # Find contours
        contours, _ = cv2.findContours(slice_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Filter and keep only large contours
        filtered_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

        # Draw filtered contours onto the cleaned mask
        cv2.drawContours(cleaned_mask[i], filtered_contours, -1, (1), thickness=cv2.FILLED)

    return cleaned_mask

# Example usage
min_contour_area = 1000  # Adjust based on your dataset
cleaned_rnfl = remove_small_contours(rnfl, min_contour_area)

# Print confirmation
print("Unique values after cleaning:", np.unique(cleaned_rnfl))
print("Shape of cleaned RNFL:", np.shape(cleaned_rnfl))

Unique values after cleaning: [0 1]
Shape of cleaned RNFL: (73, 200, 400)


In [101]:
@interact(x=(0,len(cleaned_rnfl)-1,1))
def h(x=0):
    colorimage = process_label(cleaned_rnfl[x, :,:])
    
    f = plt.figure(figsize=(6,6))
    plt.imshow(colorimage)
    plt.axis('off')

interactive(children=(IntSlider(value=0, description='x', max=72), Output()), _dom_classes=('widget-interact',…

In [102]:
rnfl_zoomed = ZoomNumpyVolume(cleaned_rnfl)
print(np.unique(rnfl_zoomed))
print(np.shape(rnfl_zoomed))


@interact(x=(0,len(rnfl_zoomed)-1,1))
def h(x=0):
    colorimage = process_label(rnfl_zoomed[x, :,:])
    
    f = plt.figure(figsize=(6,6))
    plt.imshow(colorimage)
    plt.axis('off')

[0 1]
(429, 200, 400)


interactive(children=(IntSlider(value=0, description='x', max=428), Output()), _dom_classes=('widget-interact'…

In [103]:
np.save('rnfl_zoomed.npy',rnfl_zoomed)

In [104]:
rnfl = np.load('rnfl_zoomed.npy')
print(np.shape(rnfl))

(429, 200, 400)


In [105]:
rnfl_slices = np.copy(rnfl)

In [106]:
@interact(x=(0,len(rnfl_slices)-1,1))
def h(x=0):
    f = plt.figure(figsize=(6,6))
    # f.add_subplot(1,3, 1)
    plt.imshow(rnfl_slices[x], cmap='gray')
    plt.show()

interactive(children=(IntSlider(value=0, description='x', max=428), Output()), _dom_classes=('widget-interact'…

In [107]:
shape_3d = np.zeros((400, 200, 400), dtype=bool)

slice_count = 0
for slices in range(0,len(rnfl_slices),5):
    binary_image = np.array(Image.fromarray(np.array(rnfl_slices[slices],dtype=np.uint8)).convert('L').resize((100,100)),dtype=np.uint8)

    for i in range(binary_image.shape[0]):
        for j in range(binary_image.shape[1]):
            if binary_image[i, j] == 1:
                shape_3d[i, j, slice_count:slice_count+1] = True
                
    slice_count = slice_count + 1

print('shape_3d created...', slice_count)   

shape_3d created... 86


In [108]:
# Convert to VoxelGrid
vgrid = VoxelGrid(shape_3d)

# Convert voxels to a mesh
mesh = vgrid.as_boxes()

# Set face color (RGBA)
orange = [255, 165, 0, 255]  # R, G, B, A
mesh.visual.face_colors = orange

mesh.show()